# Tier 2 — Re-run the analyses (no MMseqs2 needed)

**What this needs:** the code + the **Tier 2** archive (`tier2_caches.zip`, ~483 MB) on top of Tier 1. Tier 2 adds the processed corpus caches (`Cache/*.pkl`), the MMseqs2 alignment outputs (`MMseqs/**/result.m8`), and the protein-conservation outputs.

**Why:** it lets you re-run any analysis step — change a parameter, add a metric, regenerate a figure's data — **without** installing MMseqs2/BLAST/MAFFT and **without** re-parsing 40k tunes. The alignments are already computed; you work from them.

The processed `.pkl` caches also pin the exact TheSession snapshot used in the paper, so results are reproducible regardless of upstream weekly updates.

In [1]:
# --- Colab bootstrap: clone + install (+ fetch data) on Colab; no-op locally ---
import sys, os, subprocess

if "google.colab" in sys.modules:
    if not os.path.isdir("TheSessionEvo"):
        subprocess.run(["git", "clone",
                        "https://github.com/jomimc/TheSessionEvo.git"], check=True)
    os.chdir("TheSessionEvo")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
    r = subprocess.run([sys.executable, "scripts/download_data.py",
                        "--tier", "full"])
    if r.returncode:
        print("\nData download failed. Set the archive URLs in"
              " scripts/download_data.py (pending the published record),"
              " or unzip the tier archives at the repo root.")
else:
    print("Not on Colab \u2014 using the local checkout.")


Not on Colab — using the local checkout.


In [2]:
import sys
from pathlib import Path


def find_repo_root(start=None):
    """Walk upward until we find the Src/thesession package."""
    p = Path(start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / "Src" / "thesession").is_dir():
            return cand
    raise RuntimeError("Could not locate the repo root (Src/thesession not found).")


REPO = find_repo_root()
if str(REPO / "Src") not in sys.path:
    sys.path.insert(0, str(REPO / "Src"))
print("Repo root:", REPO)


Repo root: /home/jmcbride/projects/TheSessionEvo


## 1. Make sure the Tier 2 data is present

```bash
python scripts/download_data.py --tier full
```

(Downloads Tier 1 + Tier 2. Until the Zenodo record exists, unzip both `Zenodo/*.zip` at the repo root.)

In [3]:
def have(*rel_paths):
    """True if every path (relative to the repo root) exists."""
    return all((REPO / p).exists() for p in rel_paths)

tier2 = ["Cache/thesession_cleaned_processed.pkl",
         "Cache/all_parts_thesession_df.pkl",
         "MMseqs/thesession_parts/result.m8"]
if have(*tier2):
    print("Tier 2 data present \u2014 good to go.")
else:
    print("Tier 2 data NOT found. Run:  python scripts/download_data.py --tier full")

Tier 2 data present — good to go.


## 2. Load the corpus objects

These are the seven objects every analysis works from. With `redo=False` and the Tier 2 caches in place, they load from disk in a minute or two — no parsing, no MMseqs2:

| object | what it is |
|---|---|
| `df` | one row per cleaned tune setting |
| `tunes` | parsed `music21` streams, keyed by setting id |
| `df_parts` | one row per tune *part* (A/B section) |
| `parts_data` | per-part pitch-class sequences + metadata |
| `res` | pairwise alignment hits between similar parts |
| `res0`, `mismatches` | annotated hits + per-position substitutions |

In [4]:
from thesession.io import tune_loader as load_tunes, seq_io
from thesession.structure import part_separation as PS
from thesession.alignment import parts as PA

df, tunes = load_tunes.load_thesession_data(redo=False)
df_parts, parts_data = PS.get_all_parts_thesession(df, tunes, redo=False)
res = seq_io.load_mmseqs_pairwise(df, "thesession_parts", annotate=False)
res = PA.prune_identical_parts(res, parts_data)
res, res0, mismatches = PA.annotate_res(df, df_parts, res, parts_data, redo=False)

print(f"tunes:        {len(df):,} settings")
print(f"parts:        {len(df_parts):,} tune parts")
print(f"aligned pairs:{len(res):,} part-vs-part hits")

tunes:        39,978 settings
parts:        50,935 tune parts
aligned pairs:3,644,204 part-vs-part hits


## 3. Explore

`df` is an ordinary pandas DataFrame — slice and inspect it however you like.

In [5]:
df[[c for c in ['name', 'type', 'meter', 'mode'] if c in df.columns]].head()

,name,type,meter,mode
0,'G Iomain Nan Gamhna,slip jig,9/8,Gmajor
1,'G Iomain Nan Gamhna,slip jig,9/8,Amixolydian
2,'S Ann An Ìle,strathspey,4/4,Gmajor
4,'S Daor An Tabac,reel,4/4,Bminor
5,'S Iomadh Rud A Chunnaic Mi,reel,4/4,Gmajor


## 4. Re-run an analysis from the cached alignments

Each `data_for_<stage>(...)` function recomputes the numerical data behind a figure **from the loaded objects above** — i.e. from the cached alignments, with no MMseqs2 call. Change a scoring parameter in `Src/thesession/pipeline/substitution.py`, re-run with `redo=True`, then re-render the figure to see the effect.

This is left behind a `RUN_HEAVY` flag because it recomputes over every aligned pair (a few minutes). Flip it to `True` to run it.

In [6]:
RUN_HEAVY = False   # set True to recompute Fig 3's data from the cached alignments

if RUN_HEAVY:
    from thesession.pipeline.substitution import data_for_substitution
    from thesession.viz import main_figs
    import matplotlib; matplotlib.use("Agg")

    data_for_substitution(df, tunes, df_parts, parts_data, res, res0, mismatches, redo=True)
    main_figs.fig3()
    from IPython.display import Image, display
    display(Image(filename=str(REPO / "Figures" / "fig3.png")))
else:
    print("RUN_HEAVY is False \u2014 skipping the recompute. Set it to True to run it.")

RUN_HEAVY is False — skipping the recompute. Set it to True to run it.


### Regenerate *everything* from the caches

To rebuild every figure's data and re-render all figures from the Tier 2 caches in one call (still no MMseqs2, since the alignments are cached):

```python
import run_pipeline
run_pipeline.main(redo=False)   # redo=True forces a full recompute of everything
```

With `redo=False` this just re-derives anything missing and re-renders; with `redo=True` it recomputes all figure data from the cached alignments (longer, but still no MMseqs2).